# Init modules

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import col, trim

In [0]:
RENAME_MAP = {
    'cst_id': 'customer_id',
    'cst_key': 'customer_key',
    'cst_firstname': 'firstname',
    'cst_lastname': 'lastname',
    'cst_marital_status': 'marital_status',
    'cst_gndr': 'gender',
    'cst_create_date': 'create_date'
}

# Reading From Bronze

In [0]:
df = spark.table('baraa_dev_project.bronze.crm_cust_info')

# Data Transformations

## Trim the string values

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

## Normalization for martial_status, gndr

In [0]:
df = (
    df
    .withColumn(
        'cst_marital_status',
        F.when(F.upper(F.col('cst_marital_status')) == 'S', 'Single')
        .when(F.upper(F.col('cst_marital_status')) == 'M', 'Married')
        .otherwise("N/A")
    )
    .withColumn(
        'cst_gndr',
        F.when(F.upper(F.col('cst_gndr')) == 'M', 'Male')
        .when(F.upper(F.col('cst_gndr')) == 'F', 'Female')
        .otherwise("N/A")
    )
)

## Rename not friendly column names

In [0]:
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)


# Write Into Silver Table

In [0]:
(
    df.write
        .mode('overwrite')
        .format('delta')
        .saveAsTable('baraa_dev_project.silver.crm_customers')
)